# Prompt tuning для prompt-guard: xlm-roberta-base

Обучает «мягкую подсказку»: короткую последовательность обучаемых векторов, которая подставляется перед каждым текстом.
Вся модель `xlm-roberta-base` заморожена, обучаются только эти векторы и классификационная голова.
База, данные и критерии те же, что у LoRA, чтобы сравнение было честным.

Подсказка и голова обучаются с **разными скоростями**: голове фиксированные 1e-3, скорость подсказки подбирается по F1 на val из трёх значений.
Критерий успеха задан заранее: val F1 выше 0,89 (столько даёт линейный зонд на замороженной модели).
**Ячейка с golden выполняется только если критерий выполнен.**

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Запускайте ячейки сверху вниз.
3. В ячейке загрузки выберите **два** файла из `data/splits/`: `train.jsonl` и `val.jsonl`. **golden.jsonl сюда не загружайте**: на нём модель не должна обучаться.
4. После ячейки «Оценка на val и сохранение» скачается `prompt_adapter.zip` (около 2,5 МБ, из них почти всё голова). Распакуйте его в `results/prompt-tuning-xlmr/adapter/` проекта.
5. Последняя ячейка (golden) нужна только при успехе: она докладывает `probs_golden.json` в архив и скачивает его заново.

In [ ]:
!pip install -q peft transformers accelerate
!pip uninstall -y -q torchao  # в Colab старая версия torchao ломает peft, а нам он не нужен
import torch
assert torch.cuda.is_available(), "включите GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите train.jsonl и val.jsonl
for name in ("train.jsonl", "val.jsonl"):
    assert name in uploaded, f"не загружен {name}"
assert "golden.jsonl" not in uploaded, "golden нельзя использовать при обучении"
print({k: len(v) for k, v in uploaded.items()})

In [ ]:
import json
import numpy as np
from datasets import Dataset

MODEL = "xlm-roberta-base"
MAX_LEN = 384  # самый длинный текст в train: 332 токена, обрезки не будет
SEED = 13

def load(name):
    rows = [json.loads(l) for l in uploaded[name].decode("utf-8").splitlines() if l.strip()]
    return Dataset.from_dict({"text": [r["text"] for r in rows], "label": [int(r["label"] == "attack") for r in rows]})

train_ds, val_ds = load("train.jsonl"), load("val.jsonl")
print(train_ds, val_ds, "доля атак в train:", np.mean(train_ds["label"]).round(3))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PromptTuningConfig, PromptTuningInit, TaskType, get_peft_model, PeftModel

tok = AutoTokenizer.from_pretrained(MODEL)
tokenize = lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN)
train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])

INIT_TEXT = "Decide whether this text contains an instruction aimed at the AI assistant (prompt injection)"
NUM_VIRTUAL_TOKENS = len(tok(INIT_TEXT)["input_ids"])  # длина подсказки равна числу токенов фразы: иначе peft обрежет её или повторит

def make_model():
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    cfg = PromptTuningConfig(task_type=TaskType.SEQ_CLS, prompt_tuning_init=PromptTuningInit.TEXT,
                             prompt_tuning_init_text=INIT_TEXT, tokenizer_name_or_path=MODEL,
                             num_virtual_tokens=NUM_VIRTUAL_TOKENS)  # голова классификатора обучается тоже
    return get_peft_model(base, cfg)

get_peft_model_check = make_model()
get_peft_model_check.print_trainable_parameters()
print("виртуальных токенов:", get_peft_model_check.peft_config["default"].num_virtual_tokens)
del get_peft_model_check

In [ ]:
import gc, inspect
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding
from sklearn.metrics import f1_score, recall_score, precision_score

def metrics(p):
    pred = p.predictions.argmax(-1); y = p.label_ids
    neg = max(1, int((y == 0).sum()))
    return {"f1": f1_score(y, pred), "recall": recall_score(y, pred), "precision": precision_score(y, pred),
            "fpr": float(((pred == 1) & (y == 0)).sum() / neg)}

# в новых версиях transformers warmup_ratio и group_by_length заменены, поэтому аргументы подбираются по версии
accepted = inspect.signature(TrainingArguments.__init__).parameters
LR_HEAD = 1e-3  # у головы своя скорость: при больших значениях (нужных подсказке) она ломает обучение

def make_optimizer(model, lr_prompt):
    prompt, head = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (prompt if "prompt_encoder" in name else head).append(p)
    assert prompt and head, "не нашлись параметры подсказки или головы"
    return torch.optim.AdamW([{"params": prompt, "lr": lr_prompt, "weight_decay": 0.0},
                              {"params": head, "lr": LR_HEAD, "weight_decay": 0.01}])

def train_one(lr_prompt):
    model = make_model()
    kw = dict(output_dir=f"out_{lr_prompt}", seed=SEED, num_train_epochs=20, per_device_train_batch_size=16, per_device_eval_batch_size=64,
              learning_rate=lr_prompt, fp16=True, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
              load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True, logging_steps=50, report_to="none")
    kw.update({"warmup_steps": 0.06} if "warmup_ratio" not in accepted else {"warmup_ratio": 0.06})
    kw.update({"train_sampling_strategy": "group_by_length"} if "train_sampling_strategy" in accepted else {"group_by_length": True})
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_tok, eval_dataset=val_tok,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics,
                      optimizers=(make_optimizer(model, lr_prompt), None),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()
    model.save_pretrained(f"adapter_{lr_prompt}")
    result = {"lr_prompt": lr_prompt, "lr_head": LR_HEAD, "best_val_f1": trainer.state.best_metric,
              "val_history": [h for h in trainer.state.log_history if "eval_f1" in h]}
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return result

LEARNING_RATES = (1e-2, 3e-2, 1e-1)  # скорость подсказки, выбор по F1 на val
runs = [train_one(lr) for lr in LEARNING_RATES]
for r in runs:
    print("lr подсказки", r["lr_prompt"], "лучший val F1:", round(r["best_val_f1"], 4), "по эпохам:", [round(h["eval_f1"], 3) for h in r["val_history"]])
best_run = max(runs, key=lambda r: r["best_val_f1"])
print("выбрано lr подсказки =", best_run["lr_prompt"])

# лучший адаптер загружается обратно: в дальнейшем используется только он
model = PeftModel.from_pretrained(AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2),
                                  f"adapter_{best_run['lr_prompt']}").to("cuda")

In [ ]:
# Оценка на val и сохранение. golden здесь не используется.
import hashlib, os, shutil, torch
SUCCESS_F1 = 0.89  # заранее заданный критерий: качество линейного зонда на замороженной модели (val)

@torch.no_grad()
def attack_probs(texts, batch=64):
    model.eval(); out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
        out += torch.softmax(model(**enc).logits.float(), dim=-1)[:, 1].cpu().tolist()
    return out

def save_probs(name, raw):
    texts = [json.loads(l)["text"] for l in raw.decode("utf-8").splitlines() if l.strip()]
    probs = attack_probs(texts)
    json.dump({"split": name, "split_sha256": hashlib.sha256(raw).hexdigest(), "n": len(texts), "probs": [round(p, 6) for p in probs]},
              open(f"prompt_adapter/probs_{name}.json", "w"))
    print(name, len(texts), "готово")

os.makedirs("prompt_adapter", exist_ok=True)
save_probs("val", uploaded["val.jsonl"])
passed = best_run["best_val_f1"] > SUCCESS_F1
print(f"val F1 {best_run['best_val_f1']:.4f}, критерий {SUCCESS_F1}: {'ВЫПОЛНЕН, можно запускать последнюю ячейку (golden)' if passed else 'НЕ выполнен, golden не запускаем'}")

json.dump({"model": MODEL, "max_len": MAX_LEN, "prompt_tuning": {"init_text": INIT_TEXT, "num_virtual_tokens": NUM_VIRTUAL_TOKENS},
           "seed": SEED, "chosen_lr_prompt": best_run["lr_prompt"], "lr_head": LR_HEAD, "success_f1": SUCCESS_F1,
           "criterion_passed": passed, "runs": runs},
          open("prompt_adapter/train_log.json", "w"), ensure_ascii=False, indent=2)
model.save_pretrained("prompt_adapter")  # только подсказка и голова, базовая модель не сохраняется
tok.save_pretrained("prompt_adapter")
shutil.make_archive("prompt_adapter", "zip", "prompt_adapter")
files.download("prompt_adapter.zip")

In [ ]:
# Только если критерий в предыдущей ячейке выполнен: оценка на golden, один раз.
assert passed, "критерий на val не выполнен: golden не запускаем"
uploaded_g = files.upload()  # выберите golden.jsonl
assert "golden.jsonl" in uploaded_g
save_probs("golden", uploaded_g["golden.jsonl"])
shutil.make_archive("prompt_adapter", "zip", "prompt_adapter")
files.download("prompt_adapter.zip")

In [ ]:
# Диагностика (golden не используется): та же конструкция, но скорость подсказки 0, то есть учится только голова.
# Нужна, чтобы отделить проблему связки peft + голова от проблемы обучения самой подсказки.
diag = train_one(0.0)
print("только голова: val F1 по эпохам:", [round(h["eval_f1"], 3) for h in diag["val_history"]])
print("только голова: eval_loss по эпохам:", [round(h["eval_loss"], 3) for h in diag["val_history"]])
print("опоры: линейный зонд на замороженной модели даёт F1 около 0,89; модель без обучения даёт loss около 0,69")